# Paraphrasing Project DeFan

In [ ]:
import pandas as pd

# --------------------------------------------------
# load original dataset
# --------------------------------------------------

df = pd.read_csv("defan_cleaned.csv")

# --------------------------------------------------
# answers with exactly 15 questions
# --------------------------------------------------

answer_counts = df.groupby("answer")["question"].nunique()
answers_15 = answer_counts[answer_counts == 15].index

df15 = df[df["answer"].isin(answers_15)].copy()

# --------------------------------------------------
# build answer-group table
# --------------------------------------------------

groups = (
    df15.groupby("answer")
        .first()[["domain", "type"]]
        .reset_index()
)

# --------------------------------------------------
# determine how many groups per domain
# --------------------------------------------------

target_groups = 1000
domain_counts = groups["domain"].value_counts()
domains = domain_counts.index.tolist()

# base allocation
base = target_groups // len(domains)
quota = {d: min(base, domain_counts[d]) for d in domains}

# fill remaining slots
remaining = target_groups - sum(quota.values())

for d in domains:
    if remaining == 0:
        break
    spare = domain_counts[d] - quota[d]
    add = min(spare, remaining)
    quota[d] += add
    remaining -= add

# --------------------------------------------------
# sample answer groups
# --------------------------------------------------

selected_answers = []

for d in domains:
    ans = groups[groups["domain"] == d]["answer"]
    chosen = ans.sample(quota[d], random_state=0)
    selected_answers.extend(chosen.tolist())

# --------------------------------------------------
# build final dataset
# --------------------------------------------------

final_df = df15[df15["answer"].isin(selected_answers)].copy()

answer_to_class = {
    ans: f"C{i:04d}"
    for i, ans in enumerate(sorted(selected_answers), start=1)
}

final_df["class_id"] = final_df["answer"].map(answer_to_class)
final_df = final_df.sort_values(["class_id", "question"]).reset_index(drop=True)
final_df["question_in_class"] = final_df.groupby("class_id").cumcount() + 1

final_df = final_df[
    ["class_id", "question_in_class", "question", "answer", "domain", "type"]
]

# --------------------------------------------------
# export
# --------------------------------------------------

final_df.to_json(
    "defan_15q_15000_balanced.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [16]:
final_df.head(15000)

,class_id,question_in_class,question,answer,domain,type
0,C0001,1,EMNLP 2016 location,"Austin, Texas, United States",Conference venue,city
1,C0001,2,Give me the venue for 2016 EMNLP,"Austin, Texas, United States",Conference venue,city
2,C0001,3,"In 2016, where was EMNLP held?","Austin, Texas, United States",Conference venue,city
3,C0001,4,"In 2016, where was EMNLP hosted?","Austin, Texas, United States",Conference venue,city
4,C0001,5,What location accommodated EMNLP in 2016?,"Austin, Texas, United States",Conference venue,city
...,...,...,...,...,...,...
14995,C1000,11,Who was granted the Nobel Prize for Peace in 1...,Óscar Arias,Nobel Prize,name
14996,C1000,12,Who was honored with the Nobel Prize for Peace...,Óscar Arias,Nobel Prize,name
14997,C1000,13,Who was the Nobel Prize recipient for Peace in...,Óscar Arias,Nobel Prize,name
14998,C1000,14,Who was the laureate of the Nobel Prize for Pe...,Óscar Arias,Nobel Prize,name


In [13]:
final_df["domain"].value_counts()

domain
Nobel Prize            8655
entertainment          4035
World Organizations     930
Conference venue        795
Sports                  585
Name: count, dtype: int64

In [15]:
final_df[final_df["domain"]=="Sports"]

,class_id,question_in_class,question,answer,domain,type
600,C0041,1,How many fans in the stands watched the final ...,"107,412",Sports,numeric
601,C0041,2,How many fans were in the stands for the final...,"107,412",Sports,numeric
602,C0041,3,How many individuals attended the final match ...,"107,412",Sports,numeric
603,C0041,4,How many people attended the final match of th...,"107,412",Sports,numeric
604,C0041,5,How many people turned up for 1970 FIFA World ...,"107,412",Sports,numeric
...,...,...,...,...,...,...
14365,C0958,11,Which stadium acted as the host for the 1966 F...,Wembley Stadium,Sports,name
14366,C0958,12,Which stadium hosted the 1966 FIFA World Cup f...,Wembley Stadium,Sports,name
14367,C0958,13,Which stadium played host to the decisive matc...,Wembley Stadium,Sports,name
14368,C0958,14,Which stadium served as the location for the f...,Wembley Stadium,Sports,name
